In [1]:
!python --version

Python 3.14.5


The system cannot find the path specified.


## ___World Checklist of Vascular Plants (WCVP) homotypic synonyms___
----------------

In [24]:
from collections import namedtuple

import pandas as pd
import numpy as np

In [3]:
# look up https://diatoms.org/news/faq-what-are-homotypic-and-heterotypic-synonyms for homotypic synonyms vs heterotypic synonyms

In [4]:
# World Checklist of Vascular Plants (WCVP) dataset from => https://sftp.kew.org/pub/data-repositories/WCVP/

wcvp = pd.read_csv(r"../../data/chapter2/wcvp/wcvp_names.csv", low_memory=False, sep='|', true_values=['T'])

WCVP_COLUMNS = [
    "taxon_name", # full binominal name
    "plant_name_id", # World Checklist of Vascular Plants (WCVP) identifier
    "accepted_plant_name_id", # the ID of the accepted name of this taxon. Where the taxon_status is "Accepted", this will be identical to the plant_name_id value. May be empty if taxon status is unplaced, ilegitimate, or in some cases where the accepted name is not a vascular plant (e.g. a moss, alga or animal).
    "basionym_plant_name_id", # ID of the original name that taxon_name was derived from. If there is a parenthetical author it is a basionym. If there is a replaced synonym author it is a replaced synonym. If empty there have been no name changes. 
    "taxon_status", # indication of nomenclatural status and taxonomic opinion regarding the name
    "homotypic_synonym" # boolan indicating if the name is a homotypic synonym
]

In [5]:
# only pick the records that are homotypic synonyms; i.e. (taxon_status=="Synonym") and (homotypic_synonym==True)

synonym_pairs = wcvp.loc[:, WCVP_COLUMNS].query(r"taxon_status=='Synonym' and homotypic_synonym and (plant_name_id!=accepted_plant_name_id)").loc[:, ("taxon_name", "plant_name_id", "accepted_plant_name_id")]
synonym_pairs = synonym_pairs.astype({"plant_name_id": int, "accepted_plant_name_id": int}).reset_index(drop=True)

In [6]:
# replace the WCVP identifiers with binominal names

wcvp_synonyms_lookup_table = pd.merge(left=synonym_pairs, left_on="accepted_plant_name_id", right=wcvp.loc[:, WCVP_COLUMNS], right_on="plant_name_id", how="left", suffixes=("_syn", '')).loc[:, ("taxon_name", "taxon_name_syn")]
wcvp_synonyms_lookup_table

,taxon_name,taxon_name_syn
0,Caladenia minorata,Caladenia glossodia
1,Chiloglottis gunnii,Caladenia gunnii
2,Volkameria acerbiana,Clerodendrum acerbianum
3,Veronica sibthorpioides,Cochlidiosperma sibthorpioides
4,Veronica sibthorpioides,Veronica hederifolia subsp. sibthorpioides
...,...,...
281047,Wedelia subpetiolata,Aspilia subpetiolata
281048,Wedelia tomentosa,Aspilia tomentosa
281049,Wedelia trichostephia,Seruneum trichostephia
281050,Wedelia trichostephia,Trichostemma hispidum


In [7]:
wcvp_synonyms_lookup_table.rename({"taxon_name": "name", "taxon_name_syn": "synonym"}, axis=1)

,name,synonym
0,Caladenia minorata,Caladenia glossodia
1,Chiloglottis gunnii,Caladenia gunnii
2,Volkameria acerbiana,Clerodendrum acerbianum
3,Veronica sibthorpioides,Cochlidiosperma sibthorpioides
4,Veronica sibthorpioides,Veronica hederifolia subsp. sibthorpioides
...,...,...
281047,Wedelia subpetiolata,Aspilia subpetiolata
281048,Wedelia tomentosa,Aspilia tomentosa
281049,Wedelia trichostephia,Seruneum trichostephia
281050,Wedelia trichostephia,Trichostemma hispidum


In [8]:
wcvp_synonyms_lookup_table.rename({"taxon_name": "name", "taxon_name_syn": "synonym"}, axis=1).to_csv(r"../../data/chapter2/wcvp/synonyms.csv", index=False)

In [25]:
synonym = namedtuple(typename="synonym", field_names=["name", "synonym"])

In [33]:
# redo the synonym extraction for the final subset - some species are mising synonyms even though the WCVP dataset has synonyms for them

final = pd.read_csv(r"../../data/chapter2/FRED/subsets/final.csv")
names = pd.read_csv(r"../../data/chapter2/FRED/subsets/final.csv").binominal

In [34]:
wcvp_synonyms_lookup_table.query(r"taxon_name.isin(@names)")

,taxon_name,taxon_name_syn
116,Metrosideros umbellata,Agalmanthus umbellata
264,Inga ruiziana,Feuilleea ruiziana
479,Heptapleurum heptaphyllum,Aralia heptaphylla
603,Galium aparine,Asterophyllum aparine
693,Syzygium acuminatissimum,Acmena acuminatissima
...,...,...
280952,Pappobolus microphyllus,Helianthus microphyllus
280966,Helianthus praecox,Helianthus debilis subsp. praecox
280967,Helianthus praecox,Helianthus cucumerifolius var. praecox
280969,Helianthus radula,Helianthus atrorubens subsp. radula


In [35]:
synonyms = [synonym(name=name, synonym=tuple(df.taxon_name_syn.values)) for (name, df) in wcvp_synonyms_lookup_table.query(r"taxon_name.isin(@names)").groupby("taxon_name", as_index=False)]

In [36]:
wcvp_synonyms_lookup_table.query(r"taxon_name_syn.isin(@names)")

,taxon_name,taxon_name_syn
2485,Ragala sanguinolenta,Chrysophyllum sanguinolentum
30916,Lycopodium volubile,Pseudodiphasium volubile
50455,Blechnum discolor,Lomaria discolor
50654,Onoclea struthiopteris,Matteuccia struthiopteris
53136,Camphora officinarum,Cinnamomum camphora
53148,Camphora micrantha,Cinnamomum micranthum
53152,Camphora parthenoxylon,Cinnamomum parthenoxylon
65897,Blechnum novae-zelandiae,Parablechnum novae-zelandiae
75193,Camphora glandulifera,Cinnamomum glanduliferum
80586,Blechnum procerum,Parablechnum procerum


In [37]:
synonyms += [synonym(name=name, synonym=tuple(df.taxon_name.values)) for (name, df) in wcvp_synonyms_lookup_table.query(r"taxon_name_syn.isin(@names)").groupby("taxon_name_syn", as_index=False)]

In [38]:
len(synonyms)

848

In [43]:
synonyms = pd.DataFrame(synonyms)

In [42]:
final.synonyms.isna().sum()

np.int64(475)

In [50]:
pd.merge(left=final.drop("synonyms", axis=1), left_on="binominal", right=synonyms, right_on="name", how="left").drop("name", axis=1)

,binominal,F01286,F01289,F00679,F00727,F00709,state,state_source,synonym
0,Strobilanthes dimorphotricha,Strobilanthes,Acanthaceae,-1.313788,4.915894,-2.006791,AM,FRED4,"(Goldfussia dimorphotricha,)"
1,Actinidia kolomikta,Actinidia,Actinidiaceae,-1.469676,5.413982,-2.227362,AM,FRED4,"(Prunus kolomikta, Trochostigma kolomikta)"
2,Saurauia herthae,Saurauia,Actinidiaceae,-0.691449,3.557723,-1.911707,AM,"(Brundrett and Tedersoo, 2018)",NaN
3,Liquidambar formosana,Liquidambar,Altingiaceae,-1.189440,4.180184,-1.409904,AM,FRED4,NaN
4,Liquidambar gracilipes,Liquidambar,Altingiaceae,-0.333180,1.505939,0.027494,AM,"(Brundrett and Tedersoo, 2018)","(Altingia gracilipes,)"
...,...,...,...,...,...,...,...,...,...
1296,Dioon rzedowskii,Dioon,Zamiaceae,-0.316577,2.799503,-1.439027,AM,FRED4,NaN
1297,Encephalartos gratus,Encephalartos,Zamiaceae,-0.010640,1.975898,-1.544651,AM,FRED4,NaN
1298,Zamia lucayana,Zamia,Zamiaceae,-1.116225,4.075373,-1.515606,AM,FRED4,NaN
1299,Alpinia japonica,Alpinia,Zingiberaceae,-1.778152,5.505626,-1.727221,AM,FRED4,"(Globba japonica, Languas japonica)"


In [51]:
pd.merge(left=final.drop("synonyms", axis=1), left_on="binominal", right=synonyms, right_on="name", how="left").drop("name", axis=1).synonym.isna().sum()

np.int64(453)

In [53]:
pd.merge(left=final.drop("synonyms", axis=1), left_on="binominal", right=synonyms, right_on="name", how="left").drop("name", axis=1).to_csv(r"../../data/chapter2/FRED/subsets/final_name_matched.csv", index=False)